# MNIST Neural Network from Scratch with NumPy

This notebook implements a small fully-connected neural network for handwritten digit classification using **NumPy**.

TensorFlow is used only to load the MNIST dataset. The neural network, forward pass, loss, backpropagation, and parameter updates are implemented manually.

## Architecture

```text
28 × 28 image
     ↓
784 input values
     ↓
Dense Layer 1: 784 → 128
     ↓
ReLU
     ↓
Dense Layer 2: 128 → 10
     ↓
Softmax
     ↓
10 probabilities (digits 0–9)
```

Training follows:

**Forward pass → Loss → Backward pass → Parameter update → Repeat**


## 1. Imports

NumPy handles the numerical operations. TensorFlow's MNIST loader gives us the dataset.


In [1]:
import numpy as np
from tensorflow.keras.datasets import mnist


I0000 00:00:1791017576.222964   16399 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791017576.223325   16399 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791017576.256734   16399 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791017576.993886   16399 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

## 2. Load MNIST

MNIST contains 60,000 training images and 10,000 test images.

Each image is `28 × 28`, giving:

`28 × 28 = 784` pixels.

Pixel values are stored from `0` to `255`. We later divide by `255` to normalize them to approximately `0` to `1`.


In [2]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", X_test.shape)
print("Test labels:", y_test.shape)


Training images: (60000, 28, 28)
Training labels: (60000,)
Test images: (10000, 28, 28)
Test labels: (10000,)


## 3. Model architecture

### Layer 1: 784 → 128

Every input pixel connects to every hidden neuron.

- `W1`: `(784, 128)`
- `b1`: `(1, 128)`

### Layer 2: 128 → 10

Every hidden neuron connects to each output neuron.

- `W2`: `(128, 10)`
- `b2`: `(1, 10)`

The 10 output neurons correspond to digits `0` through `9`.


## 4. Layer 1 — initialize parameters

Weights start as small random values. Biases start at zero.

These weights and biases are the parameters the network learns.


In [3]:
# Layer 1: 784 inputs → 128 hidden neurons
W1 = np.random.randn(784, 128) * 0.01
b1 = np.zeros((1, 128))


## 5. Layer 2 — initialize parameters


In [4]:
# Layer 2: 128 hidden neurons → 10 output neurons
W2 = np.random.randn(128, 10) * 0.01
b2 = np.zeros((1, 10))

lr = 0.01


## 6. Softmax

The second layer produces 10 raw scores called **logits**.

Softmax converts those scores into probabilities:

- each probability is between 0 and 1
- all 10 probabilities add up to 1

Subtracting the largest logit before exponentiation makes the calculation numerically stable.


In [5]:
def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


# Forward Pass

The forward pass turns input images into predictions.

```text
X
 ↓
Z1 = XW1 + b1
 ↓
A1 = ReLU(Z1)
 ↓
Z2 = A1W2 + b2
 ↓
Softmax
 ↓
probabilities
```

We keep the intermediate values because backpropagation needs them.


## Forward Pass — Layer 1

Each `28 × 28` image is flattened to 784 values and normalized.

The dense layer computes:

`Z1 = X @ W1 + b1`

Then ReLU introduces non-linearity:

`A1 = max(0, Z1)`


In [6]:
batch_size = 32

X = X_train[:batch_size].reshape(-1, 784) / 255.0
y = y_train[:batch_size]

# Layer 1 forward pass
Z1 = X @ W1 + b1
A1 = np.maximum(0, Z1)


## Forward Pass — Layer 2

Layer 2 converts the 128 hidden activations into 10 output logits.


In [7]:
# Layer 2 forward pass
Z2 = A1 @ W2 + b2
probabilities = softmax(Z2)


## 7. Cross-entropy loss

For each image, the label tells us which of the 10 probabilities is correct.

If the correct digit is `7`, we select the probability at class index `7`.

For one example:

`L = -log(correct probability)`

High correct probability → small loss.

Low correct probability → large loss.

For a batch, we average the losses.


In [8]:
N = len(X)

correct_probabilities = probabilities[np.arange(N), y]
loss = -np.mean(np.log(correct_probabilities))

print("Loss:", loss)


Loss: 2.302201581571331


# Backward Pass

Backpropagation works backward from the loss.

Its job is to calculate how much each parameter contributed to the loss.

The gradient tells us how changing a parameter would change the loss.

```text
Loss
 ↓
dZ2
 ↓
dW2, db2
 ↓
dA1
 ↓
dZ1
 ↓
dW1, db1
```

For softmax combined with cross-entropy, the output gradient simplifies to:

`dZ2 = probabilities - one_hot_labels`

Because our batch loss is an average, we divide by `N`.


## Backward Pass — Layer 2

First calculate the gradient entering Layer 2.

Then calculate how that gradient affects `W2` and `b2`.


In [9]:
# Gradient entering Layer 2
dZ2 = probabilities.copy()
dZ2[np.arange(N), y] -= 1
dZ2 /= N

# Gradients for Layer 2 parameters
dW2 = A1.T @ dZ2
db2 = np.sum(dZ2, axis=0, keepdims=True)


## Backward Pass — Layer 1

The gradient now travels backward through Layer 2 and then through ReLU.

For ReLU:

- derivative = `1` when `Z1 > 0`
- derivative = `0` when `Z1 <= 0`

Finally we calculate the gradients for `W1` and `b1`.


In [10]:
# Pass gradient backward through Layer 2
dA1 = dZ2 @ W2.T

# Pass gradient through ReLU
dZ1 = dA1 * (Z1 > 0)

# Gradients for Layer 1 parameters
dW1 = X.T @ dZ1
db1 = np.sum(dZ1, axis=0, keepdims=True)


# Parameter Update

Gradient descent updates each parameter using:

`parameter = parameter - learning_rate × gradient`

The learning rate controls the size of each step.


In [11]:
W1 -= lr * dW1
b1 -= lr * db1

W2 -= lr * dW2
b2 -= lr * db2


# Complete Training Loop

A **batch** contains 32 images.

One **epoch** means the network has processed all 60,000 training images once.

For every batch:

1. Prepare the images.
2. Forward pass.
3. Calculate loss.
4. Backpropagate.
5. Update parameters.

We repeat this for several epochs.


In [19]:
# Start from fresh parameters for a clean training run
W1 = np.random.randn(784, 128) * 0.01
b1 = np.zeros((1, 128))

W2 = np.random.randn(128, 10) * 0.01
b2 = np.zeros((1, 10))

lr = 0.01
batch_size = 32
epochs = 5

for epoch in range(epochs):
    total_loss = 0

    for start in range(0, len(X_train), batch_size):
        end = start + batch_size

        X = X_train[start:end].reshape(-1, 784) / 255.0
        y = y_train[start:end]
        N = len(X)

        # ---------- Forward: Layer 1 ----------
        Z1 = X @ W1 + b1
        A1 = np.maximum(0, Z1)

        # ---------- Forward: Layer 2 ----------
        Z2 = A1 @ W2 + b2
        probabilities = softmax(Z2)

        # ---------- Loss ----------
        correct_probabilities = probabilities[np.arange(N), y]
        loss = -np.mean(np.log(correct_probabilities))
        total_loss += loss * N

        # ---------- Backward: Layer 2 ----------
        dZ2 = probabilities.copy()
        dZ2[np.arange(N), y] -= 1
        dZ2 /= N

        dW2 = A1.T @ dZ2
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        # ---------- Backward: Layer 1 ----------
        dA1 = dZ2 @ W2.T
        dZ1 = dA1 * (Z1 > 0)

        dW1 = X.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # ---------- Update ----------
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

    average_loss = total_loss / len(X_train)
    print(f"Epoch {epoch + 1}: Loss = {average_loss:.4f}")


Epoch 1: Loss = 1.2484
Epoch 2: Loss = 0.4293
Epoch 3: Loss = 0.3485
Epoch 4: Loss = 0.3133
Epoch 5: Loss = 0.2888


# Testing on an Unseen Image

The test set was not used to update the parameters.

We run one unseen image through the same forward pass and use `argmax` to choose the class with the highest probability.


In [20]:
correct = 0

for i in range(len(X_test)):

    # Prepare image
    X = X_test[i].reshape(1, 784) / 255.0

    # Forward pass
    Z1 = X @ W1 + b1
    A1 = np.maximum(0, Z1)

    Z2 = A1 @ W2 + b2
    probabilities = softmax(Z2)

    # Prediction
    prediction = np.argmax(probabilities)

    # Compare with actual label
    if prediction == y_test[i]:
        correct += 1

accuracy = correct / len(X_test)

print("Correct:", correct)
print("Total:", len(X_test))
print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

Correct: 9234
Total: 10000
Accuracy: 0.9234
Accuracy (%): 92.34


# Key Takeaways

The core training mechanism is:

```text
Input
  ↓
Forward pass
  ↓
Prediction
  ↓
Loss
  ↓
Backpropagation
  ↓
Gradients
  ↓
Parameter update
  ↓
Repeat
```

You implemented:

- Dense layers
- ReLU
- Softmax
- Cross-entropy loss
- Backpropagation
- Gradient descent
- Mini-batch training
- MNIST digit classification

This is the same fundamental training loop used inside much larger neural networks; larger models mainly add more layers, more parameters, different architectures, and more sophisticated optimization.
